# Exploratory Data Analysis & Leakage Detection

This notebook performs comprehensive EDA on loan data to:
- Understand data distribution and quality
- Detect potential data leakage
- Identify key features for modeling
- Assess class imbalance

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')
%matplotlib inline

## 1. Load Data

In [ ]:
data_path = Path('../data/raw')

csv_files = list(data_path.glob('*.csv'))
if csv_files:
    df = pd.read_csv(csv_files[0], low_memory=False)
    print(f"Loaded: {csv_files[0]}")
else:
    print("No CSV file found. Please run data download script first.")
    df = None

In [ ]:
if df is not None:
    print(f"Dataset shape: {df.shape}")
    print(f"\nColumns: {df.columns.tolist()[:20]}...")  
    df.head()

## 2. Data Quality Assessment

In [ ]:
if df is not None:
    missing_data = pd.DataFrame({
        'Column': df.columns,
        'Missing_Count': df.isnull().sum(),
        'Missing_Percent': (df.isnull().sum() / len(df) * 100).round(2)
    }).sort_values('Missing_Percent', ascending=False)
    
    print("Top 20 columns with missing data:")
    print(missing_data.head(20))

In [ ]:
if df is not None:
    high_missing = missing_data[missing_data['Missing_Percent'] > 50]
    print(f"Columns with >50% missing: {len(high_missing)}")
    print("\nThese should be removed:")
    print(high_missing['Column'].tolist())

## 3. Target Variable Analysis

In [ ]:
if df is not None and 'loan_status' in df.columns:
    print("Loan Status Distribution:")
    print(df['loan_status'].value_counts())
    print(f"\nUnique values: {df['loan_status'].nunique()}")

In [ ]:
if df is not None and 'loan_status' in df.columns:
    df['default'] = df['loan_status'].apply(
        lambda x: 1 if x in ['Charged Off', 'Default', 'Late (31-120 days)'] else 0
    )
    
    fig, ax = plt.subplots(1, 2, figsize=(14, 5))
    
    df['default'].value_counts().plot(kind='bar', ax=ax[0])
    ax[0].set_title('Default Distribution (Binary)')
    ax[0].set_xlabel('Default (0=No, 1=Yes)')
    ax[0].set_ylabel('Count')
    
    default_rate = df['default'].mean()
    ax[1].pie([1-default_rate, default_rate], labels=['No Default', 'Default'], 
              autopct='%1.1f%%', startangle=90)
    ax[1].set_title('Default Rate')
    
    plt.tight_layout()
    plt.show()
    
    print(f"\nDefault rate: {default_rate:.2%}")
    print(f"Class imbalance ratio: {(1-default_rate)/default_rate:.2f}:1")

## 4. Leakage Detection

Critical step: Identify features that contain information not available at prediction time.

In [ ]:
if df is not None:
    leakage_keywords = [
        'total_pymnt', 'recoveries', 'collection_recovery',
        'last_pymnt', 'next_pymnt', 'debt_settlement_flag',
        'settlement', 'hardship'
    ]
    
    leakage_features = []
    for col in df.columns:
        col_lower = col.lower()
        for keyword in leakage_keywords:
            if keyword in col_lower:
                leakage_features.append(col)
                break
    
    print(f"Potential leakage features found: {len(leakage_features)}")
    print("\nThese features should be removed:")
    for feat in leakage_features[:20]:
        print(f"  - {feat}")

## 5. Feature Distributions

In [ ]:
if df is not None:
    key_features = ['loan_amnt', 'int_rate', 'annual_inc', 'dti', 'fico_range_high']
    available_features = [f for f in key_features if f in df.columns]
    
    if available_features:
        fig, axes = plt.subplots(2, 3, figsize=(15, 10))
        axes = axes.flatten()
        
        for idx, feature in enumerate(available_features[:6]):
            df[feature].hist(bins=50, ax=axes[idx], edgecolor='black')
            axes[idx].set_title(f'{feature} Distribution')
            axes[idx].set_xlabel(feature)
            axes[idx].set_ylabel('Frequency')
        
        for idx in range(len(available_features), 6):
            axes[idx].axis('off')
        
        plt.tight_layout()
        plt.show()

## 6. Correlation Analysis

In [ ]:
if df is not None and 'default' in df.columns:
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    sample_cols = numeric_cols[:20]
    
    corr_with_target = df[sample_cols].corrwith(df['default']).sort_values(ascending=False)
    
    print("Top 10 features correlated with default:")
    print(corr_with_target.head(10))
    
    print("\nBottom 10 features (negative correlation):")
    print(corr_with_target.tail(10))

In [ ]:
if df is not None:
    sample_features = ['loan_amnt', 'int_rate', 'annual_inc', 'dti', 'revol_bal', 'default']
    available = [f for f in sample_features if f in df.columns]
    
    if len(available) > 1:
        plt.figure(figsize=(10, 8))
        correlation_matrix = df[available].corr()
        sns.heatmap(correlation_matrix, annot=True, fmt='.2f', cmap='coolwarm', center=0)
        plt.title('Feature Correlation Matrix')
        plt.tight_layout()
        plt.show()

## 7. Feature Engineering Opportunities

In [ ]:
if df is not None:
    if 'loan_amnt' in df.columns and 'annual_inc' in df.columns:
        df['loan_to_income'] = df['loan_amnt'] / (df['annual_inc'] + 1)
        print("Created: loan_to_income ratio")
    
    if 'int_rate' in df.columns and df['int_rate'].dtype == 'object':
        df['int_rate_clean'] = df['int_rate'].str.rstrip('%').astype(float)
        print("Cleaned: interest rate")
    
    if 'term' in df.columns:
        df['term_months'] = df['term'].str.extract('(\d+)').astype(float)
        print("Created: term in months")

## 8. Summary & Next Steps

In [ ]:
if df is not None:
    print("="*60)
    print("EDA SUMMARY")
    print("="*60)
    print(f"Total records: {len(df):,}")
    print(f"Total features: {len(df.columns)}")
    print(f"\nDefault rate: {df['default'].mean():.2%}")
    print(f"Leakage features identified: {len(leakage_features)}")
    print(f"High-missing columns (>50%): {len(high_missing)}")
    print("\nNext steps:")
    print("1. Remove leakage features")
    print("2. Handle missing values")
    print("3. Engineer features")
    print("4. Build baseline model")
    print("5. Optimize advanced model")